In [ ]:
from py_module.metadata.connection.ConnectionRegistry import PostgresConnection as pgconn
from py_module.metadata.render_jinja.RenderRegistry import ChangeDataCaptureExtraction as cdc, RetrieveSchema as rs, SchemaSource as ss 
from py_module.metadata.datatype_conversion.avro import DataTypeConverter as dtc
from sqlalchemy import text
import json
from fastavro import parse_schema
import os


# SETUP (connection + get template)

In [2]:
params = {
    "database":"postgres",
    "db": "meteo",
    "host": "localhost",
    "port": 5433,
    "user": "meteo",
    "password": "meteo",
    "ssl_args": {}
} 

In [3]:
engine = pgconn.get_engine(**params)
conn = engine.connect()

2025-10-26 09:33:40 - [INFO   ] | PostgreSQL engine created successfully


In [66]:
cdc_extraction = cdc.render_jinja(database=params['database'])
retrieve_schema = rs.render_jinja(database=params['database'])
schema_source = ss.render_jinja()

# Render template with appropriate values

### Retrieve schema table -- this is useful for next steps

In [126]:
table_schema = 'public'
table_name = None
bucket_name = 'postgres__d-meteo-db'

rendered_schema = retrieve_schema.render(
                                table_schema=table_schema,
                                table_name=table_name
                            )
converter = dtc()


In [127]:
base_dir = os.path.join(os.getcwd(),'source_db_schema')

In [128]:
it_schema = conn.execute(text(rendered_schema))

In [129]:
tables = it_schema.fetchall()
dist_datasets = list(set(i[0] for i in tables))
dist_tables = list(set(i[1] for i in tables))

In [136]:
for d in dist_datasets:
    for t in dist_tables:

        tmp_records = [col for col in tables if col[0] == str(d) and col[1] == str(t)]
        target_dir = os.path.join(base_dir,params['database'],params['db'],d)

        cdc_columns = []
        avro_columns = []
        dbt_columns = []

        for c in tmp_records:
            col_name = c[2]
            db_type = c[3]
            precision = c[4]
            scale = c[5]

            avro_type = converter.source_to_avro(params['database'], db_type, numeric_precision=precision, numeric_scale=scale)
            bq_type = converter.source_to_bigquery(params['database'], db_type)

            # setup for cdc model injection
            cdc_columns.append(col_name)

            # setup the columns for avro injection with default
            avro_field = converter.generate_avro_field(col_name, avro_type)
            avro_columns.append(avro_field)

            # setup the columns for dbt source
            dbt_columns.append({col_name: bq_type})

        dbt_source_model = schema_source.render(
            schema_name = d, 
            table_name = t, 
            staging_dataset = 'staging', 
            database = params['database'], 
            istance_name = params['db'],
            bucket_name = 'postgres__d-meteo-db',
            version = 'v1', 
            cols = dbt_columns
        )

        avro_schema = {
            "name": f"{d}_{t}__record",
            "type": "record",
            "fields": avro_columns
        }

        parsed_schema = parse_schema(avro_schema)
        
        os.makedirs(target_dir,exist_ok=True)

        with open(os.path.join(target_dir,f'{d}__{t}.yml'),'w') as f:
            f.write(dbt_source_model)
                
        with open(os.path.join(target_dir,f'{d}__{t}_avro.json'), "w", encoding="utf-8") as f:
            json.dump(parsed_schema, f, indent=2)

        
            

2025-10-26 21:38:33 - [WARNING ] | No mapping for postgres.name, defaulting to ['null','string']
2025-10-26 21:38:33 - [WARNING ] | No mapping for postgres.name, defaulting to ['null','string']
2025-10-26 21:38:33 - [WARNING ] | No mapping for postgres.name, defaulting to ['null','string']
2025-10-26 21:38:33 - [WARNING ] | No mapping for postgres.name, defaulting to ['null','string']
2025-10-26 21:38:33 - [WARNING ] | No mapping for postgres.name, defaulting to ['null','string']
2025-10-26 21:38:33 - [WARNING ] | No mapping for postgres.name, defaulting to ['null','string']
2025-10-26 21:38:33 - [WARNING ] | No mapping for postgres.name, defaulting to ['null','string']
2025-10-26 21:38:33 - [WARNING ] | No mapping for postgres.name, defaulting to ['null','string']


2025-10-26 21:38:33 - [WARNING ] | No mapping for postgres.name, defaulting to ['null','string']
2025-10-26 21:38:33 - [WARNING ] | No mapping for postgres.name, defaulting to ['null','string']
2025-10-26 21:38:33 - [WARNING ] | No mapping for postgres.name, defaulting to ['null','string']
2025-10-26 21:38:33 - [WARNING ] | No mapping for postgres.name, defaulting to ['null','string']
2025-10-26 21:38:33 - [WARNING ] | No mapping for postgres.name, defaulting to ['null','string']
2025-10-26 21:38:33 - [WARNING ] | No mapping for postgres.name, defaulting to ['null','string']


In [ ]:
converter = dtc()  # your DataTypeConverter instance, optionally pass defaults

dbt_columns = list()
avro_columns = list()
cdc_columns = list()

for i in it_schema:
    col_name = i[0]
    db_type = i[1]
    precision = i[2]
    scale = i[3]

    avro_type = converter.source_to_avro(params['database'], db_type, numeric_precision=precision, numeric_scale=scale)
    bq_type = converter.source_to_bigquery(params['database'], db_type)

    # setup for cdc model injection
    cdc_columns.append(col_name)

    # setup the columns for avro injection with default
    avro_field = converter.generate_avro_field(col_name, avro_type)
    avro_columns.append(avro_field)

    # setup the columns for dbt source
    dbt_columns.append({col_name: bq_type})

TypeError: PostgresConnection.get_engine() takes 1 positional argument but 2 were given